In [1]:
import duckdb

con = duckdb.connect("../warehouse.duckdb", read_only=True)

def q(sql):
    return con.execute(sql).fetchdf()

In [2]:
import pandas as pd

rows = []
tables = q("SELECT table_name FROM information_schema.tables WHERE table_schema = 'raw'")["table_name"]

for t in tables:
    total = q(f"SELECT COUNT(*) AS n FROM raw.{t}")["n"][0]
    cols = q(f"""SELECT column_name FROM information_schema.columns
                 WHERE table_schema = 'raw' AND table_name = '{t}'""")["column_name"]
    for c in cols:
        r = q(f'SELECT COUNT(DISTINCT "{c}") AS d, COUNT(*) - COUNT("{c}") AS nulls FROM raw.{t}').iloc[0]
        rows.append((t, c, total, int(r["d"]), int(r["nulls"])))

prof = pd.DataFrame(rows, columns=["table", "column", "rows", "distinct", "nulls"])
prof["candidate_key"] = (prof["distinct"] == prof["rows"]) & (prof["nulls"] == 0)

prof[prof["candidate_key"]]

,table,column,rows,distinct,nulls,candidate_key
0,customers,customer_id,99441,99441,0,True
10,orders,order_id,99441,99441,0,True
11,orders,customer_id,99441,99441,0,True
37,products,product_id,32951,32951,0,True
46,product_category_name_translation,product_category_name,71,71,0,True
47,product_category_name_translation,product_category_name_english,71,71,0,True
48,sellers,seller_id,3095,3095,0,True


In [3]:
q("""
SELECT
    COUNT(*)                            AS customer_rows,
    COUNT(DISTINCT customer_id)         AS distinct_customer_id,
    COUNT(DISTINCT customer_unique_id)  AS distinct_people
FROM raw.customers
""")

,customer_rows,distinct_customer_id,distinct_people
0,99441,99441,96096


In [4]:
q("""
SELECT orders_per_person, COUNT(*) AS people
FROM (
    SELECT c.customer_unique_id, COUNT(*) AS orders_per_person
    FROM raw.orders o
    JOIN raw.customers c USING (customer_id)
    GROUP BY 1
)
GROUP BY 1
ORDER BY 1
""")

,orders_per_person,people
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


In [5]:
q("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - (SELECT COUNT(*) FROM (SELECT DISTINCT order_id, review_id FROM raw.order_reviews)) AS duplicate_rows
FROM raw.order_reviews
""")

,total_rows,duplicate_rows
0,99224,0


In [6]:
q("""
SELECT
    COUNT(*)                   AS total_rows,
    COUNT(DISTINCT review_id)  AS distinct_review_id,
    COUNT(DISTINCT order_id)   AS distinct_order_id
FROM raw.order_reviews
""")

,total_rows,distinct_review_id,distinct_order_id
0,99224,98410,98673


In [7]:
q("""
SELECT
    (SELECT COUNT(*) FROM raw.orders)                                           AS orders,
    (SELECT COUNT(*) FROM raw.order_items)                                      AS items,
    (SELECT COUNT(*) FROM raw.orders o JOIN raw.order_items i USING (order_id)) AS orders_x_items,
    (SELECT COUNT(*) FROM raw.orders o
        JOIN raw.order_items i USING (order_id)
        JOIN raw.order_payments p USING (order_id))                             AS orders_x_items_x_payments
""")

,orders,items,orders_x_items,orders_x_items_x_payments
0,99441,112650,112650,117601


In [8]:
q("""
SELECT n_reviews, COUNT(*) AS orders
FROM (SELECT order_id, COUNT(*) AS n_reviews FROM raw.order_reviews GROUP BY 1)
GROUP BY 1 ORDER BY 1
""")

,n_reviews,orders
0,1,98126
1,2,543
2,3,4


In [9]:
q("""
SELECT
    ROUND((SELECT SUM(price) FROM raw.order_items), 2) AS items_revenue,
    ROUND((SELECT SUM(i.price) FROM raw.order_items i JOIN raw.order_payments p USING (order_id)), 2) AS after_payments_join
""")

,items_revenue,after_payments_join
0,13591643.7,14209115.34


In [10]:
q("""
WITH pay AS (
    SELECT order_id, SUM(payment_value) AS paid
    FROM raw.order_payments
    GROUP BY order_id
)
SELECT
    COUNT(*)                   AS rows_after_join,
    ROUND(SUM(i.price), 2)     AS items_revenue
FROM raw.order_items i
LEFT JOIN pay USING (order_id)
""")

,rows_after_join,items_revenue
0,112650,13591643.7


In [11]:
q("""
SELECT order_status, COUNT(*) AS orders
FROM raw.orders
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [12]:
q("""
SELECT
    MIN(order_purchase_timestamp) AS first_order,
    MAX(order_purchase_timestamp) AS last_order
FROM raw.orders
""")

,first_order,last_order
0,2016-09-04 21:15:19,2018-10-17 17:30:18


In [13]:
q("""
SELECT
    date_trunc('month', order_purchase_timestamp) AS month,
    COUNT(*)                                      AS orders
FROM raw.orders
GROUP BY 1
ORDER BY 1
""")

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [14]:
q("""
SELECT o.order_status,
       COUNT(DISTINCT o.order_id) AS orders,
       COUNT(i.order_id)          AS item_rows,
       ROUND(SUM(i.price), 2)     AS item_revenue
FROM raw.orders o
LEFT JOIN raw.order_items i USING (order_id)
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders,item_rows,item_revenue
0,delivered,96478,110197,13221498.11
1,shipped,1107,1185,150727.44
2,canceled,625,542,95235.27
3,unavailable,609,7,2007.69
4,invoiced,314,359,61526.37
5,processing,301,357,60439.22
6,created,5,0,NaN
7,approved,2,3,209.60


In [15]:
q("""
SELECT order_status,
       COUNT(*)                            AS orders,
       COUNT(order_delivered_customer_date) AS with_delivery_date
FROM raw.orders
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders,with_delivery_date
0,delivered,96478,96470
1,shipped,1107,0
2,canceled,625,6
3,unavailable,609,0
4,invoiced,314,0
5,processing,301,0
6,created,5,0
7,approved,2,0


In [16]:
q("""
SELECT o.order_status,
       COUNT(*) AS orders,
       COUNT(*) FILTER (WHERE i.order_id IS NULL) AS orders_without_items
FROM raw.orders o
LEFT JOIN (SELECT DISTINCT order_id FROM raw.order_items) i ON o.order_id = i.order_id
GROUP BY 1
ORDER BY 2 DESC
""")

,order_status,orders,orders_without_items
0,delivered,96478,0
1,shipped,1107,1
2,canceled,625,164
3,unavailable,609,603
4,invoiced,314,2
5,processing,301,0
6,created,5,5
7,approved,2,0


In [17]:
q("""
SELECT
    COUNT(*) FILTER (WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL) AS delivered_no_date,
    COUNT(*) FILTER (WHERE order_delivered_customer_date < order_purchase_timestamp)            AS delivered_before_purchase,
    COUNT(*) FILTER (WHERE order_estimated_delivery_date IS NULL)                               AS no_estimate
FROM raw.orders
""")

,delivered_no_date,delivered_before_purchase,no_estimate
0,8,0,0


In [18]:
q("SELECT ROUND(SUM(price), 2) AS price_total, ROUND(SUM(freight_value), 2) AS freight_total FROM raw.order_items")

,price_total,freight_total
0,13591643.7,2251909.54


In [19]:
q("""
SELECT
    COUNT(*)                                                                  AS delivered_with_date,
    COUNT(*) FILTER (WHERE order_delivered_customer_date > order_estimated_delivery_date)                   AS late_timestamp,
    COUNT(*) FILTER (WHERE CAST(order_delivered_customer_date AS DATE) > CAST(order_estimated_delivery_date AS DATE)) AS late_date,
    COUNT(*) FILTER (WHERE CAST(order_estimated_delivery_date AS TIME) <> TIME '00:00:00') AS estimate_has_time
FROM raw.orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""")

,delivered_with_date,late_timestamp,late_date,estimate_has_time
0,96470,7826,6534,0


In [20]:
q("""
SELECT
    COUNT(*)                                          AS orders_with_multiple_reviews,
    COUNT(*) FILTER (WHERE min_score <> max_score)    AS with_differing_scores
FROM (
    SELECT order_id, MIN(review_score) AS min_score, MAX(review_score) AS max_score
    FROM raw.order_reviews
    GROUP BY 1
    HAVING COUNT(*) > 1
)
""")

,orders_with_multiple_reviews,with_differing_scores
0,547,202


In [21]:
q("""
SELECT
    *
    FROM raw.order_reviews
    LIMIT 10

""")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
5,15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,NaN,NaN,2018-04-13,2018-04-16 00:39:37
6,07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,NaN,NaN,2017-07-16,2017-07-18 19:30:34
7,7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,NaN,NaN,2018-08-14,2018-08-14 21:36:06
8,a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,NaN,NaN,2017-05-17,2017-05-18 12:05:37
9,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelh...,2018-05-22,2018-05-23 16:45:47


In [22]:
q("""
SELECT date_trunc('month', order_purchase_timestamp) AS month, COUNT(*) AS shipped_orders
FROM raw.orders
WHERE order_status = 'shipped'
GROUP BY 1 ORDER BY 1
""")

,month,shipped_orders
0,2016-09-01,1
1,2016-10-01,8
2,2017-01-01,16
3,2017-02-01,21
4,2017-03-01,45
5,2017-04-01,49
6,2017-05-01,55
7,2017-06-01,47
8,2017-07-01,56
9,2017-08-01,41


In [23]:
q("""
SELECT
  COUNT(*) FILTER (WHERE late_days BETWEEN 1 AND 7) AS late_1_to_7_days,
  COUNT(*) FILTER (WHERE late_days > 7)             AS late_over_7_days
FROM (
  SELECT date_diff('day', CAST(order_estimated_delivery_date AS DATE),
                          CAST(order_delivered_customer_date AS DATE)) AS late_days
  FROM raw.orders
  WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
)
""")

,late_1_to_7_days,late_over_7_days
0,3672,2862


In [24]:
q("""
SELECT
    COUNT(*) AS products,
    COUNT(*) FILTER (WHERE p.product_category_name IS NULL) AS no_category,
    COUNT(*) FILTER (WHERE p.product_category_name IS NOT NULL
                       AND t.product_category_name IS NULL) AS no_translation
FROM raw.products p
LEFT JOIN raw.product_category_name_translation t
       ON p.product_category_name = t.product_category_name
""")

,products,no_category,no_translation
0,32951,610,13


In [25]:
q("""
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name, 'NO CATEGORY') AS category,
    COUNT(*) AS item_rows,
    ROUND(SUM(i.price), 2) AS revenue
FROM raw.order_items i
JOIN raw.orders o USING (order_id)
JOIN raw.products p USING (product_id)
LEFT JOIN raw.product_category_name_translation t
       ON p.product_category_name = t.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY 1
ORDER BY 3 DESC
""")

,category,item_rows,revenue
0,health_beauty,9465,1233131.72
1,watches_gifts,5859,1166176.98
2,bed_bath_table,10953,1023434.76
3,sports_leisure,8431,954852.55
4,computers_accessories,7644,888724.61
...,...,...,...
69,flowers,33,1110.04
70,home_comfort_2,30,760.27
71,cds_dvds_musicals,14,730.00
72,fashion_childrens_clothes,7,519.95


In [26]:
q("""
SELECT CAST(order_purchase_timestamp AS DATE) AS day, COUNT(*) AS orders
FROM raw.orders
WHERE order_purchase_timestamp >= DATE '2017-11-15'
  AND order_purchase_timestamp <  DATE '2017-12-05'
GROUP BY 1
ORDER BY 1
""")

,day,orders
0,2017-11-15,186
1,2017-11-16,226
2,2017-11-17,197
3,2017-11-18,149
4,2017-11-19,158
5,2017-11-20,230
6,2017-11-21,228
7,2017-11-22,201
8,2017-11-23,283
9,2017-11-24,1176


### Metrics validation

In [27]:
q("""
SELECT ROUND(SUM(i.price) / COUNT(DISTINCT o.order_id), 2) AS aov
FROM raw.order_items i JOIN raw.orders o USING (order_id)
WHERE o.order_status = 'delivered'
""")

,aov
0,137.04


In [28]:
q("""
SELECT COUNT(*) AS rows, ROUND(AVG(review_score), 3) AS avg_score
FROM (
    SELECT order_id, review_score
    FROM raw.order_reviews
    QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_answer_timestamp DESC, review_id) = 1
)
""")

,rows,avg_score
0,98673,4.086


In [29]:
q("""
WITH per_person AS (
    SELECT c.customer_unique_id, COUNT(*) AS delivered_orders
    FROM raw.orders o JOIN raw.customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
    GROUP BY 1
)
SELECT
    COUNT(*)                                              AS customers,
    COUNT(*) FILTER (WHERE delivered_orders >= 2)         AS repeat_customers,
    ROUND(100.0 * COUNT(*) FILTER (WHERE delivered_orders >= 2) / COUNT(*), 2) AS repeat_rate_pct
FROM per_person
""")

,customers,repeat_customers,repeat_rate_pct
0,93358,2801,3.0


In [30]:
q("""
SELECT ROUND(AVG(date_diff('day', order_purchase_timestamp, order_delivered_customer_date)), 2) AS avg_days
FROM raw.orders
WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""")

,avg_days
0,12.5


In [31]:
q("""
SELECT COUNT(*) AS rows, ROUND(AVG(review_score), 3) AS avg_score
FROM (
    SELECT r.order_id, r.review_score
    FROM raw.order_reviews r JOIN raw.orders o USING (order_id)
    WHERE o.order_status = 'delivered'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY r.order_id ORDER BY r.review_answer_timestamp DESC, r.review_id) = 1
)
""")

,rows,avg_score
0,95832,4.156


In [32]:
q("""
SELECT
  COUNT(*)                                                              AS orders_placed,
  COUNT(*) FILTER (WHERE order_status = 'delivered')                    AS delivered,
  COUNT(*) FILTER (WHERE order_status IN ('shipped','invoiced','processing','approved','created')) AS in_flight,
  COUNT(*) FILTER (WHERE order_status = 'canceled')                     AS canceled,
  ROUND(100.0 * COUNT(*) FILTER (WHERE order_status = 'canceled') / COUNT(*), 2) AS cancel_rate_pct
FROM raw.orders
""")

,orders_placed,delivered,in_flight,canceled,cancel_rate_pct
0,99441,96478,1729,625,0.63


In [33]:
q("""
SELECT CAST(order_purchase_timestamp AS DATE) AS day, COUNT(*) AS orders
FROM raw.orders
WHERE order_purchase_timestamp >= DATE '2018-08-15'
  AND order_purchase_timestamp <  DATE '2018-09-15'
GROUP BY 1 ORDER BY 1
""")

,day,orders
0,2018-08-15,288
1,2018-08-16,320
2,2018-08-17,257
3,2018-08-18,198
4,2018-08-19,204
5,2018-08-20,256
6,2018-08-21,243
7,2018-08-22,187
8,2018-08-23,144
9,2018-08-24,99


In [34]:
q("""
SELECT COUNT(*) AS orders,
       COUNT(*) FILTER (WHERE sellers > 1) AS multi_seller_orders
FROM (SELECT order_id, COUNT(DISTINCT seller_id) AS sellers FROM raw.order_items GROUP BY 1)
""")

,orders,multi_seller_orders
0,98666,1278


In [35]:
q("""
WITH o AS (
    SELECT c.customer_unique_id, o.order_purchase_timestamp AS ts,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp, o.order_id) AS n
    FROM raw.orders o JOIN raw.customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
)
SELECT
    COUNT(*)                                        AS repeat_customers,
    COUNT(*) FILTER (WHERE days = 0)                AS same_day,
    COUNT(*) FILTER (WHERE days BETWEEN 1 AND 30)   AS d1_30,
    COUNT(*) FILTER (WHERE days BETWEEN 31 AND 90)  AS d31_90,
    COUNT(*) FILTER (WHERE days BETWEEN 91 AND 180) AS d91_180,
    COUNT(*) FILTER (WHERE days > 180)              AS over_180
FROM (
    SELECT date_diff('day', CAST(a.ts AS DATE), CAST(b.ts AS DATE)) AS days
    FROM o a JOIN o b ON a.customer_unique_id = b.customer_unique_id AND a.n = 1 AND b.n = 2
)
""")

,repeat_customers,same_day,d1_30,d31_90,d91_180,over_180
0,2801,829,587,492,413,480


In [39]:

q("""
WITH latest AS (
    SELECT order_id, review_score
    FROM raw.order_reviews
    QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY review_answer_timestamp DESC, review_id) = 1
),
seller_orders AS (
    SELECT DISTINCT i.seller_id, i.order_id
    FROM raw.order_items i JOIN raw.orders o USING (order_id)
    WHERE o.order_status = 'delivered'
),
per_seller AS (
    SELECT s.seller_id, COUNT(l.order_id) AS reviewed_orders
    FROM seller_orders s LEFT JOIN latest l USING (order_id)
    GROUP BY 1
)
SELECT
    COUNT(*) AS sellers,
    COUNT(*) FILTER (WHERE reviewed_orders >= 5)  AS ge_5,
    COUNT(*) FILTER (WHERE reviewed_orders >= 10) AS ge_10,
    COUNT(*) FILTER (WHERE reviewed_orders >= 20) AS ge_20,
    COUNT(*) FILTER (WHERE reviewed_orders >= 30) AS ge_30,
    COUNT(*) FILTER (WHERE reviewed_orders >= 50) AS ge_50
FROM per_seller
""")


,sellers,ge_5,ge_10,ge_20,ge_30,ge_50
0,2970,1761,1236,800,623,418


In [63]:
q("""
SELECT i.order_id, COUNT(*) AS items, COUNT(DISTINCT i.seller_id) AS sellers
FROM raw.order_items i JOIN raw.orders o USING (order_id)
WHERE o.order_status = 'delivered'
GROUP BY 1 HAVING COUNT(DISTINCT i.seller_id) >= 2
LIMIT 3
""")

,order_id,items,sellers
0,002f98c0f7efd42638ed6100ca699b42,2,2
1,00bcee890eba57a9767c7b5ca12d3a1b,4,2
2,013a98b3a668bcef05b98898177f6923,2,2


In [65]:
oid = "002f98c0f7efd42638ed6100ca699b42"
for t in ["order_items", "order_payments", "order_reviews"]:
    display(q(f"SELECT * FROM raw.{t} WHERE order_id = '{oid}'"))

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,002f98c0f7efd42638ed6100ca699b42,1,d41dc2f2979f52d75d78714b378d4068,7299e27ed73d2ad986de7f7c77d919fa,2017-08-10 09:30:15,8.99,32.57
1,002f98c0f7efd42638ed6100ca699b42,2,880be32f4db1d9f6e2bec38fb6ac23ab,fa40cc5b934574b62717c68f3d678b6d,2017-08-10 09:30:15,44.90,7.16


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,002f98c0f7efd42638ed6100ca699b42,1,credit_card,9,93.62


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,6d1c23083e960eaa5daa7505e8aa7071,002f98c0f7efd42638ed6100ca699b42,5,None,None,2017-08-08,2017-08-08 23:26:12
